In [1]:
import pandas as pd
import geopandas as gpd
import json

import sys
sys.path.append('..')
from helpers import calculate_intersection

In [2]:
with open('../config/epm_to_861_utility_map.json', 'r') as f:
    epm_to_861_utility_map = json.load(f)

with open('../config/epm_to_861_ba_map.json', 'r') as f:
    epm_to_861_ba_map = json.load(f)

eia_930_meta = (
    pd.read_excel('../data/EIA930_Reference_Tables.xlsx')
    .dropna(subset='Time Zone')
)

county_geo = gpd.read_file("../data/shapefiles/US_COUNTY_2022")
erst = (
    gpd.read_file("../data/shapefiles/Electric_Retail_Service_Territories")
    .to_crs(county_geo.crs)
)
time_zone_geo = (
    gpd.read_file('../data/shapefiles/NTAD_Time_Zones_1825923810339753067.gpkg')
    .to_crs(county_geo.crs)
)

In [ ]:
epm_utility_timezone_map = {}
for epm_respondent, utility_ids in epm_to_861_utility_map.items():
    utility_ids_str = [str(id) for id in utility_ids]
    erst_sub = erst.loc[erst.ID.isin(utility_ids_str)]

    epm_respondent_time_zones = calculate_intersection(
        erst_sub,
        time_zone_geo,
        ['zone']
    )
    epm_respondent_time_zones = epm_respondent_time_zones[['zone', 'geometry']]
    epm_respondent_time_zones['area'] = epm_respondent_time_zones['geometry'].area
    time_zone = (
        epm_respondent_time_zones.sort_values('area', ascending=False)
        .iloc[0]
        ['zone']
    )
    
    epm_utility_timezone_map[epm_respondent] = time_zone

In [ ]:
ba_tz_map = dict(zip(eia_930_meta['BA Code'], eia_930_meta['Time Zone']))

In [ ]:
epm_ba_timezone_map = {}
for epm_respondent, ba_list in epm_to_861_ba_map.items():
    timezone_list = list(set([ba_tz_map[ba] for ba in ba_list]))
    if len(timezone_list) == 1:
        epm_ba_timezone_map[epm_respondent] = timezone_list[0]

In [ ]:
epm_respondent_timezone_map = epm_utility_timezone_map | epm_ba_timezone_map

In [ ]:
with open('../config/epm_respondent_timezone_map.json', 'w') as f:
    json.dump(epm_respondent_timezone_map, f)